<p align="center">
  <a href="https://github.com/wavekat/wavekat-lab">
    <img src="https://github.com/wavekat/wavekat-brand/raw/main/assets/banners/wavekat-lab-narrow.svg" alt="WaveKat Lab">
  </a>
</p>

# 03 — Build candidate clips

Turn `utterances.parquet` (notebook 01) + `vad_probs/*.npy` (notebook
02) into `candidates.parquet` — the table of audio clips that the
later notebooks will score, label, and ship to the platform.

Per the structural rules in
[`../smart-turn/docs/03-magicdata-ramc-mining.md`](../smart-turn/docs/03-magicdata-ramc-mining.md):

- For each diarized utterance `u_i` whose next utterance `u_{i+1}` is
  a **different speaker** with a clean silence gap ≥ 0.3 s, take the
  trailing window ending at `u_i.end` → `label = 1` (end-of-turn).
- For the same `u_i` (when long enough), synthesize one **mid-
  utterance** clip ending at a VAD silence dip near the midpoint →
  `label = 0` (continuation). This keeps classes balanced by
  construction without needing word-level timestamps from RAMC.

Two VAD-aware corrections layered on top:

1. **Silence-snap.** Each cut point is nudged ±300 ms to the nearest
   frame whose speech probability stays below `SILENCE_PROB` for at
   least `MIN_SILENCE_MS`. If no such frame exists, the candidate is
   rejected — the boundary sits inside continuous speech.
2. **Overlap rejection.** For end-of-turn candidates, the gap region
   `(u_i.end, u_{i+1}.start)` must contain *only* sub-threshold
   frames. Any speech frame there means the two speakers overlap and
   the structural label is unreliable.

Hard filters (drop without recording the candidate at all):

- `[+]` overlap marker on either side of the boundary.
- `[*]` unintelligible covering > 30 % of `u_i` duration.
- `u_i` shorter than `MIN_UTT_S`.
- `u_i.speaker_id == "G00000000"` (RAMC's non-speech bucket).
- Clip duration outside `[CLIP_MIN_S, CLIP_MAX_S]`.

## Configure paths

In [ ]:
from pathlib import Path

MINING_ROOT = Path("../../datasets/smart-turn-zh-mining").resolve()
UTTERANCES_PARQUET = MINING_ROOT / "utterances.parquet"
VAD_DIR            = MINING_ROOT / "vad_probs"
CANDIDATES_PARQUET = MINING_ROOT / "candidates.parquet"

SAMPLE_RATE = 16_000
WINDOW_SIZE = 512
FRAME_S = WINDOW_SIZE / SAMPLE_RATE          # 0.032 s per VAD frame

print(f"utterances : {UTTERANCES_PARQUET.name}")
print(f"vad probs  : {VAD_DIR.name}/*.npy")
print(f"out        : {CANDIDATES_PARQUET.name}")
print(f"frame_s    : {FRAME_S:.3f}")
print("✅ paths configured")

## Filter and label thresholds

All thresholds in one place so they're easy to retune after the
calibration pass on 200 audited clips.

In [ ]:
# Clip window
CLIP_MAX_S = 8.0
CLIP_MIN_S = 1.0

# Speaker-change rule (end-of-turn candidate)
MIN_GAP_S      = 0.30   # silence required between u_i.end and u_{i+1}.start
MIN_UTT_S      = 0.50   # u_i shorter than this is too noisy a target
MAX_GAP_S      = 5.00   # gaps > this likely indicate session breaks

# VAD silence-snap
SILENCE_PROB    = 0.30  # frame with prob < this counts as silence
MIN_SILENCE_MS  = 100   # need this many ms of consecutive silence
TRIM_RANGE_MS   = 300   # search window around the intended cut

# VAD overlap check (end-of-turn only)
OVERLAP_PROB    = 0.50  # frame with prob > this counts as speech

# Mid-utterance continuation cut
MIN_CUT_MARGIN_S = 1.00  # cut must sit at least this far from both ends of u_i

# Hard text filters
MAX_UNINTELLIGIBLE_FRAC = 0.30  # too high -> drop u_i
NON_SPEECH_SPEAKER = "G00000000"

# Reproducibility for the random midpoint jitter on continuation cuts
import random
RNG = random.Random(0)

print("thresholds")
for k, v in {
    "CLIP_MAX_S":             CLIP_MAX_S,
    "CLIP_MIN_S":             CLIP_MIN_S,
    "MIN_GAP_S":              MIN_GAP_S,
    "MIN_UTT_S":              MIN_UTT_S,
    "MAX_GAP_S":              MAX_GAP_S,
    "SILENCE_PROB":           SILENCE_PROB,
    "MIN_SILENCE_MS":         MIN_SILENCE_MS,
    "TRIM_RANGE_MS":          TRIM_RANGE_MS,
    "OVERLAP_PROB":           OVERLAP_PROB,
    "MIN_CUT_MARGIN_S":       MIN_CUT_MARGIN_S,
    "MAX_UNINTELLIGIBLE_FRAC": MAX_UNINTELLIGIBLE_FRAC,
}.items():
    print(f"  {k:25s} {v}")
print("✅ thresholds configured")

## Load inputs

We only build candidates for sessions that have **both** a row in
`utterances.parquet` and a `.npy` in `vad_probs/`. Anything else
would either lack a transcript (no label source) or lack VAD probs
(no silence-snap / overlap check).

In [ ]:
import pandas as pd

utt = pd.read_parquet(UTTERANCES_PARQUET)
vad_sessions = {p.stem for p in VAD_DIR.glob("*.npy") if not p.name.endswith(".tmp.npy")}
utt_sessions = set(utt["session_id"].unique())

ready = sorted(utt_sessions & vad_sessions)
missing_vad = sorted(utt_sessions - vad_sessions)
missing_utt = sorted(vad_sessions - utt_sessions)

print(f"utt sessions       : {len(utt_sessions)}")
print(f"vad sessions       : {len(vad_sessions)}")
print(f"both (will mine)   : {len(ready)}")
print(f"missing vad probs  : {len(missing_vad)} (run notebook 02 to fill)")
print(f"missing utterances : {len(missing_utt)} (re-run notebook 01)")
if missing_vad[:3]:
    for s in missing_vad[:3]:
        print(f"  no vad: {s}")
print("✅ inputs joined")

## Helpers

Three small functions that each candidate-build runs through:

- `nudge_to_silence` — given a target time and the session's VAD
  probs, walk outward in 32 ms steps to find the nearest frame whose
  probability stays below `SILENCE_PROB` for at least
  `MIN_SILENCE_MS`. Returns the snapped time in seconds, or `None`.
- `gap_is_silent` — every VAD frame strictly between `t0` and `t1`
  must be below `OVERLAP_PROB`. Catches the case where two speakers
  overlap inside what RAMC's text-only diarization called a gap.
- `truncate_text_proportional` — RAMC's TXT lacks word-level
  timestamps, so for mid-utterance continuation cuts we approximate
  the truncated transcript by character fraction. Noisy, but the
  later LLM-verification step is what filters the noise.

In [ ]:
import numpy as np


def time_to_frame(t_s: float) -> int:
    return int(round(t_s / FRAME_S))


def frame_to_time(f: int) -> float:
    return f * FRAME_S


def nudge_to_silence(probs: np.ndarray, target_s: float,
                     trim_range_ms: int = TRIM_RANGE_MS,
                     silence_prob: float = SILENCE_PROB,
                     min_silence_ms: int = MIN_SILENCE_MS) -> float | None:
    """Snap `target_s` to the nearest frame at the start of a silence run.

    Walks outward from the target frame in alternating +/- steps and
    returns the first frame whose forward window of `min_silence_ms`
    is sub-threshold. Returns `None` if no such frame exists within
    `trim_range_ms`.
    """
    target_f = time_to_frame(target_s)
    range_f = max(1, int(round(trim_range_ms / 1000 / FRAME_S)))
    silence_run = max(1, int(round(min_silence_ms / 1000 / FRAME_S)))
    n = len(probs)

    def silent_at(f: int) -> bool:
        if f < 0 or f + silence_run > n:
            return False
        return bool(np.all(probs[f : f + silence_run] < silence_prob))

    for offset in range(0, range_f + 1):
        for sign in (1, -1) if offset > 0 else (1,):
            f = target_f + sign * offset
            if silent_at(f):
                return frame_to_time(f)
    return None


def gap_is_silent(probs: np.ndarray, t0_s: float, t1_s: float,
                  overlap_prob: float = OVERLAP_PROB) -> bool:
    """True if every VAD frame strictly inside (t0, t1) is below threshold."""
    f0 = time_to_frame(t0_s)
    f1 = time_to_frame(t1_s)
    if f1 <= f0:
        return True
    f0 = max(0, f0)
    f1 = min(len(probs), f1)
    if f1 <= f0:
        return True
    return bool(np.all(probs[f0:f1] < overlap_prob))


def truncate_text_proportional(text: str, frac: float) -> str:
    if not text:
        return text
    frac = max(0.0, min(1.0, frac))
    n = max(1, int(round(frac * len(text))))
    return text[:n]


# Quick sanity check on `nudge_to_silence`
_p = np.zeros(200, dtype=np.float32)
_p[80:120] = 0.95           # a chunk of speech in the middle
snap_s = nudge_to_silence(_p, target_s=100 * FRAME_S, trim_range_ms=2_000)
snap_f = time_to_frame(snap_s) if snap_s is not None else None
print(f"synthetic probs[80:120]=0.95, target_frame=100, snapped={snap_f}")
assert snap_f is not None and (snap_f <= 80 - 0 or snap_f >= 120), "should fall outside speech run"
print("✅ helpers verified")

## Build candidates

Walk every session, look at every adjacent utterance pair, apply
filters → silence-snap → overlap check, emit at most one
end-of-turn + one continuation per qualifying parent utterance.

We track per-utterance reject reasons in a counter so the calibration
pass can see which rule is doing the most filtering.

In [ ]:
from collections import Counter
from tqdm.auto import tqdm


def utt_passes_hard_filters(row) -> str | None:
    """Return reject reason or None if utterance is eligible."""
    if row["speaker_id"] == NON_SPEECH_SPEAKER:
        return "non_speech_speaker"
    if row["duration_s"] < MIN_UTT_S:
        return "too_short"
    if row["has_overlap"]:
        return "overlap_marker"
    if row["has_unintelligible"]:
        # MAX_UNINTELLIGIBLE_FRAC is a per-utterance heuristic — we can't
        # measure char fraction without ASR per-word timestamps, so any
        # `[*]` over a *short* utterance is treated as dominant.
        if row["duration_s"] < 2 * MIN_UTT_S:
            return "unintelligible_dominant"
    return None


def make_clip_window(end_s: float, parent_start_s: float,
                     max_s: float = CLIP_MAX_S) -> tuple[float, float]:
    start_s = max(parent_start_s, end_s - max_s)
    return start_s, end_s


candidates: list[dict] = []
reject_reasons: Counter = Counter()

for session_id in tqdm(ready, desc="Sessions"):
    sess = utt[utt["session_id"] == session_id].sort_values("utt_idx").reset_index(drop=True)
    probs = np.load(VAD_DIR / f"{session_id}.npy")
    session_dur_s = len(probs) * FRAME_S

    cand_idx = 0
    for i, u in sess.iterrows():
        # ---- end-of-turn candidate (speaker_change) ----
        if i + 1 >= len(sess):
            reject_reasons["last_utt_no_neighbor"] += 1
        else:
            v = sess.iloc[i + 1]
            why_u = utt_passes_hard_filters(u)
            if why_u is not None:
                reject_reasons[f"u_filter:{why_u}"] += 1
            elif v["speaker_id"] == u["speaker_id"]:
                reject_reasons["same_speaker"] += 1
            elif v["has_overlap"]:
                reject_reasons["v_overlap_marker"] += 1
            else:
                gap = float(v["start_s"] - u["end_s"])
                if gap < MIN_GAP_S:
                    reject_reasons["gap_too_short"] += 1
                elif gap > MAX_GAP_S:
                    reject_reasons["gap_too_long"] += 1
                elif not gap_is_silent(probs, u["end_s"], v["start_s"]):
                    reject_reasons["vad_overlap_in_gap"] += 1
                else:
                    snapped_s = nudge_to_silence(probs, u["end_s"])
                    if snapped_s is None:
                        reject_reasons["no_silence_at_eot"] += 1
                    else:
                        clip_start, clip_end = make_clip_window(snapped_s, u["start_s"])
                        if clip_end - clip_start < CLIP_MIN_S:
                            reject_reasons["clip_too_short_eot"] += 1
                        else:
                            candidates.append({
                                "session_id":          session_id,
                                "candidate_idx":       cand_idx,
                                "parent_utt_idx":      int(u["utt_idx"]),
                                "source":              "speaker_change",
                                "label":               1,
                                "clip_start_s":        round(clip_start, 3),
                                "clip_end_s":          round(clip_end, 3),
                                "clip_duration_s":     round(clip_end - clip_start, 3),
                                "speaker_id":          u["speaker_id"],
                                "gender":              u["gender"],
                                "text":                u["text"],
                                "parent_text":         u["text"],
                                "gap_to_next_s":       round(gap, 3),
                                "next_speaker_id":     v["speaker_id"],
                                "vad_trim_offset_ms":  round(1000 * (snapped_s - u["end_s"]), 1),
                            })
                            cand_idx += 1

        # ---- continuation candidate (intra-utterance cut) ----
        why_u = utt_passes_hard_filters(u)
        if why_u is not None:
            reject_reasons[f"u_filter_for_cont:{why_u}"] += 1
            continue
        if u["duration_s"] < 2 * MIN_CUT_MARGIN_S:
            reject_reasons["utt_too_short_for_cut"] += 1
            continue

        cut_lo = float(u["start_s"]) + MIN_CUT_MARGIN_S
        cut_hi = float(u["end_s"])   - MIN_CUT_MARGIN_S
        target_s = RNG.uniform(cut_lo, cut_hi)
        cut_s = nudge_to_silence(probs, target_s)
        if cut_s is None or not (cut_lo <= cut_s <= cut_hi):
            reject_reasons["no_silence_in_utt"] += 1
            continue
        clip_start, clip_end = make_clip_window(cut_s, u["start_s"])
        if clip_end - clip_start < CLIP_MIN_S:
            reject_reasons["clip_too_short_cont"] += 1
            continue

        frac = (cut_s - u["start_s"]) / max(u["duration_s"], 1e-6)
        candidates.append({
            "session_id":          session_id,
            "candidate_idx":       cand_idx,
            "parent_utt_idx":      int(u["utt_idx"]),
            "source":              "intra_utterance_cut",
            "label":               0,
            "clip_start_s":        round(clip_start, 3),
            "clip_end_s":          round(clip_end, 3),
            "clip_duration_s":     round(clip_end - clip_start, 3),
            "speaker_id":          u["speaker_id"],
            "gender":              u["gender"],
            "text":                truncate_text_proportional(u["text"], frac),
            "parent_text":         u["text"],
            "gap_to_next_s":       float("nan"),
            "next_speaker_id":     "",
            "vad_trim_offset_ms":  round(1000 * (cut_s - target_s), 1),
        })
        cand_idx += 1

df = pd.DataFrame(candidates)
print(f"candidates : {len(df):,}")
print("\nreject reasons (top 15)")
for reason, count in reject_reasons.most_common(15):
    print(f"  {reason:35s} {count:>8,}")
print("✅ candidates built")

## Save

In [ ]:
if len(df) == 0:
    print("⚠️  no candidates produced — check VAD coverage and thresholds")
else:
    df = df[[
        "session_id", "candidate_idx", "parent_utt_idx", "source", "label",
        "clip_start_s", "clip_end_s", "clip_duration_s",
        "speaker_id", "gender",
        "text", "parent_text",
        "gap_to_next_s", "next_speaker_id", "vad_trim_offset_ms",
    ]]
    df.to_parquet(CANDIDATES_PARQUET, index=False)
    size_kb = CANDIDATES_PARQUET.stat().st_size / 1024
    print(f"wrote {CANDIDATES_PARQUET.name}  ({size_kb:,.1f} KiB, {len(df):,} rows)")
print("✅ candidates saved")

## Stats

Per-class yield, per-session yield, clip-duration distribution. The
calibration pass (audit 200 clips by ear) reads from this output.

In [ ]:
if len(df) == 0:
    print("no candidates — skipping stats")
else:
    print("label balance")
    print(df["label"].value_counts().sort_index().to_string())
    print("\nby source")
    print(df.groupby(["source", "label"]).size().to_string())

    print("\ncandidates per session")
    per_sess = df.groupby("session_id").size()
    print(per_sess.describe().round(1).to_string())

    print("\nclip duration (s)")
    print(df["clip_duration_s"].describe().round(2).to_string())

    print("\nVAD trim offset (ms) — eot only")
    eot = df[df["source"] == "speaker_change"]
    if len(eot):
        print(eot["vad_trim_offset_ms"].describe().round(1).to_string())

    print("\ngap-to-next (s) — eot only")
    if len(eot):
        print(eot["gap_to_next_s"].describe().round(2).to_string())
print("✅ stats printed")

## Audit a few samples

Print a handful of each label so we can eyeball the structural rule
before scaling the audit pass.

In [ ]:
if len(df) == 0:
    print("no candidates — skipping audit")
else:
    rng = np.random.default_rng(0)
    for label_value, name in [(1, "end_of_turn"), (0, "continuation")]:
        bucket = df[df["label"] == label_value]
        if not len(bucket):
            print(f"--- {name}: 0 rows ---")
            continue
        idx = rng.choice(len(bucket), size=min(5, len(bucket)), replace=False)
        sample = bucket.iloc[idx]
        print(f"\n--- {name}: {len(bucket):,} rows; showing 5 ---")
        for _, r in sample.iterrows():
            print(
                f"  {r['session_id']}  utt={r['parent_utt_idx']:4d}  "
                f"clip=[{r['clip_start_s']:.2f}, {r['clip_end_s']:.2f}]  "
                f"dur={r['clip_duration_s']:.2f}s  "
                f"src={r['source']}"
            )
            print(f"      text  : {r['text']}")
            if r["source"] == "intra_utterance_cut":
                print(f"      parent: {r['parent_text']}")
print("✅ audit printed")

## Next

`candidates.parquet` now has the `(session_id, clip_start_s,
clip_end_s, label, text)` rows the rest of the pipeline scores:

- `04_score_own_model.ipynb` — run the current zh smart-turn
  checkpoint over each clip, write per-row probability + prediction.
- `05_score_llm.ipynb` — send `text` to an LLM, write the
  `END_OF_TURN` / `CONTINUATION` verdict.
- `06_consensus_route.ipynb` — combine `label` + own-model +
  LLM, route auto-accept vs. human review.

If the audit at the bottom of this notebook surfaces obvious label
errors, retune the thresholds in the *Filter and label thresholds*
cell and re-run. The whole pipeline downstream of this is keyed on
`(session_id, candidate_idx)`, so changing labels here cleanly
invalidates the downstream parquets.